In [29]:
import torch
import os
import torchvision.transforms as T

MODEL_DIR = '/mnt/hdd/checkpoints/'
_MODELS = {
    "DINOv2-ViT-L-14": os.path.join(MODEL_DIR, "dinov2_vitl14_reg4_pretrain.pth"),
    "dinov3_vitl16": os.path.join(MODEL_DIR, 'dinov3', "dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"),
    "DINOv3.txt-ViT-L-16": os.path.join(MODEL_DIR, 'dinov3', "dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth"),
}

class DINOWrapper:
    def __init__(self, model, device):
        self.model = model
        self.model.eval()
        self.device = device
        self.preprocess = T.Compose(
            [
                T.ToTensor(),
                T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ]
        )

    @torch.no_grad()
    def encode_image(self, image):
        inputs = self.preprocess(image).to(self.device)
        if inputs.ndim==3:
            # add batch dim
            inputs = inputs.unsqueeze(0)
        feats = self.model.forward_features(inputs)
        return feats['x_norm_clstoken']

def setup_dinov3(name, device):
    model = torch.hub.load(repo_or_dir='/home/jonas/workspace/dinov3',
                                      source='local',
                                      model=name,
                                      weights=_MODELS[name])
    wrapper = DINOWrapper(model, device)
    return wrapper, wrapper.preprocess

def load_dino(name, device = "cuda" if torch.cuda.is_available() else "cpu"):
    if name not in _MODELS:
        raise RuntimeError(f"Model {name} not found in available models: {list(_MODELS.keys())}")

    if 'v3' in name:
        return setup_dinov3(name, device)

In [32]:
model, preprocess = load_dino('dinov3_vitl16')

In [31]:
from PIL import Image
img=Image.open('/mnt/hdd/datasets/CoOp/caltech-101/101_ObjectCategories/accordion/image_0001.jpg')
model.encode_image(img)

AttributeError: 'tuple' object has no attribute 'encode_image'